Setup, load environment and libs

In [9]:
from dotenv import load_dotenv
load_dotenv()  # reads .env file
import pandas as pd
import csv
import io
import os
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_mistralai import ChatMistralAI

## Build a chain

In [2]:
template = """
Does the description of this job "{description}"
fit the candidate's profile "{profile}? 
For each of the following aspects: 
* technical skills set required, 
* work area of the organization, 
* role position within the organization, 
* geographical location, 
Give me a score like: 
1.Perfect fit, 2. Good Fit, 3. Probable Fit, 4. Poor Fit, 5. Not a Fit.
make the score an integer in the range 1 to 5.
"""
prompt_template = PromptTemplate.from_template(template)

Create the analysis chain

In [3]:
def format_prompt(inputs):
    return prompt_template.format(
        description=inputs["description"], 
        profile=inputs["profile"])

format_runnable = RunnableLambda(format_prompt)

llm = ChatMistralAI(model="mistral-small-latest")  # real model, not LLM()

analysis_chain = format_runnable | llm

Create the formating chain

In [4]:
# 1. Define the structure you want extracted from the analysis
class Row(BaseModel):
    aspect: str
    score: int = Field(ge=1, le=5)
    reasoning: str

class Table(BaseModel):
    rows: list[Row]

# 2. Second model call: constrained to this exact schema
structured_llm = llm.with_structured_output(Table)

## Check with 1 example

In [5]:
job_id = 1

description = """
Job title: Senior Data Engineer Department: Data team Reports to: Lead Data Engineer 
Location: Paris, France Contract: Permanent, local and full-time Starting date: 
ASAP 2026 PUR Description PUR is a global B Corp helping companies act for climate, 
nature and people through nature-based projects. Since 2008, we have worked with 
leading companies to restore ecosystems, support farmers and communities, and 
build more resilient supply chains. Our projects include agroforestry, 
regenerative agriculture, reforestation, and ecosystem restoration, across more 
than 30 countries. We work closely with local partners, farmers, cooperatives, 
NGOs and businesses to design projects that are adapted to each landscape and 
community. This local approach is key to creating long-term impact — 
for the climate, biodiversity, soil, water, and livelihoods. Today, PUR has more 
than 200 team members across 16 countries, with key offices in Paris and Toronto. 
As the world faces growing climate and biodiversity challenges, PUR is helping 
global companies move from ambition to action. About the role As a Senior Data 
Engineer, you will design, develop, and maintain scalable data pipelines and 
data processing solutions that power analytics, reporting, and data-driven 
decision making. You will play a key role in transforming raw project data 
into reliable, high-quality datasets through robust ETL processes, modern 
data architectures, and Python-based engineering practices. Working closely with 
business and technical stakeholders, you will ensure that data is accessible, 
trustworthy, and aligned with analytical requirements while contributing to the 
continuous improvement of our data platform and cloud-based ecosystem. Your 
responsibilities will be, but not limited to: Design, develop, and maintain 
scalable ETL/ELT pipelines using Python. Build and optimize data ingestion, 
transformation, and validation processes for large datasets. Design and maintain 
data models supporting reporting, analytics, and business applications. Implement 
and monitor data quality controls to ensure reliability and accuracy. Manage and 
optimize workflows using orchestration tools such as Airflow, Prefect, or Dagster . 
Contribute to the development and maintenance of cloud-based data platforms 
(preferably Azure). Support CI/CD, testing, monitoring, and infrastructure-as-code 
practices (Terraform). Collaborate with cross-functional teams to deliver scalable 
and reliable data solutions. Maintain technical documentation and promote data 
engineering best practices. We are looking for someone with: 3-5 years of experience 
in Data Engineering , with strong Python skills and experience building 
production-grade ETL pipelines. Strong SQL expertise and solid understanding of 
relational and analytical databases, data modeling, and data warehousing concepts. 
Experience with workflow orchestration tools (Airflow, Prefect, Dagster) and 
implementing automated testing for data pipelines. Hands-on experience with cloud 
platforms (preferably Azure) and Infrastructure as Code tools (Terraform is a 
strong plus). Familiarity with AI tools (Claude is a plus). Strong knowledge of 
Git and software engineering best practices. Excellent problem-solving skills, 
focus on data quality and reliability, and ability to collaborate with 
cross-functional stakeholders. Fluent English with strong written and verbal 
communication skills. Recruitment process : Interview with the recruiter 
Interview with the manager and case study References check At PUR, we strive to 
build inclusive teams and an equitable workplace where our employees feel respected, 
supported, and empowered. As a global company operating across six continents, we 
recognize that diversity in the workplace is essential to achieving our mission. 
We welcome applications from all qualified candidates regardless of age, race, 
gender, colour, religion, national origin, sexual orientation, gender identity, 
disability, or any other aspect that makes them unique. PUR is dedicated to providing
an accessible environment for all candidates during the application process and for 
employees during their employment. If you require accommodation or consideration 
throughout the interview process, let us know, and we will do our best to support you. Find Jobs in France on Arbeitnow
"""

profile = """
I am looking for a mid to senior level Wissenschaftlicher Mitarbeiter 
(or alternatively: Informatiker, Data Scientist, Software Engineer) role in Education and Science 
(bioacoustics, ecology, climate, nature, monitoring, museum collection management, libraries), 
based in Berlin, Potsdam or Leipzig, with partly from home work arrangements. 
I bring data management (Postgres, SQL, ETL Pipelines using Airflow), Docker, research software engineering (Python, R) and machine learning (Keras, LangChain) skills, 
and I seek a position offering posibilities across research and development.
"""


In [6]:
result = analysis_chain.invoke({"description": description, "profile": profile})
print(result.content)

table = structured_llm.invoke(
    "Extract the assessment from this text, one row per aspect:\n\n" + result.content
)

Here’s the scoring for each aspect based on the candidate's profile and the job description:

1. **Technical Skills Set Required**
   - **Job Requirements:** Python, SQL, ETL pipelines, Airflow/Orchestration, Cloud (Azure), Data Modeling, CI/CD, Terraform, Git.
   - **Candidate Skills:** Postgres, SQL, ETL Pipelines (Airflow), Docker, Python, R, Machine Learning (Keras, LangChain).
   - **Score:** 3 (Good Fit) – The candidate has overlapping skills (Python, SQL, ETL, Airflow) but lacks Azure/cloud experience and infrastructure-as-code (Terraform).

2. **Work Area of the Organization**
   - **Job:** Climate, nature, and sustainability (agroforestry, reforestation, ecosystem restoration).
   - **Candidate:** Bioacoustics, ecology, climate, nature, museum collections.
   - **Score:** 2 (Good Fit) – Both focus on climate/nature, but the job is more project-based (corporate partnerships), while the candidate leans toward research/academia.

3. **Role Position Within the Organization**
   - 

Save to CSV

In [11]:
def to_csv(job_id, table, path=os.path.join("../assessments", f"{job_id}_assessment.csv")):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["JobID", "Aspect", "Score", "Reasoning"])
        for row in table.rows:
            writer.writerow([job_id, row.aspect, row.score, row.reasoning])
    return path

to_csv(job_id, table)

'../assessments/1_assessment.csv'